# How the EPA features are built

A step-by-step walkthrough on the 2025 season of what `features/epa.py` does: EPA per play for
each team's offense and defense in each game, then a **pre-game** average over previous games only.

In [ ]:
%load_ext autoreload
%autoreload 2

In [40]:
pbp = pd.read_parquet("../data/raw/pbp_2025.parquet")
print(pbp.shape)
pbp.head()

(48771, 11)


,game_id,season,week,season_type,posteam,defteam,pass,rush,epa,success,passer_player_id
0,2025_01_ARI_NO,2025,1,REG,NaN,NaN,0.0,0.0,-0.000000,0.0,NaN
1,2025_01_ARI_NO,2025,1,REG,ARI,NO,0.0,0.0,-0.352700,0.0,NaN
2,2025_01_ARI_NO,2025,1,REG,ARI,NO,0.0,1.0,-0.190052,0.0,NaN
3,2025_01_ARI_NO,2025,1,REG,ARI,NO,1.0,0.0,1.317340,1.0,00-0035228
4,2025_01_ARI_NO,2025,1,REG,ARI,NO,1.0,0.0,-1.694360,0.0,00-0035228


## 1. Keep pass and run plays, then average EPA per play for each offense

In [41]:
plays = pbp[((pbp["pass"] == 1) | (pbp["rush"] == 1)) & pbp["epa"].notna()].copy()
print(pbp.shape, plays.shape)

offense = plays.groupby(["game_id", "season", "week","posteam"]).agg(
    off_epa=("epa", "mean"),
    plays=("epa", "count"),
).reset_index()

offense.shape 
offense.head()

(48771, 11) (36155, 11)


,game_id,season,week,posteam,off_epa,plays
0,2025_01_ARI_NO,2025,1,ARI,0.061773,66
1,2025_01_ARI_NO,2025,1,NO,-0.057140,73
2,2025_01_BAL_BUF,2025,1,BAL,0.394435,51
3,2025_01_BAL_BUF,2025,1,BUF,0.256330,81
4,2025_01_CAR_JAX,2025,1,CAR,-0.176198,63


## 2. The same for each defense (EPA allowed per play)

In [42]:
plays = pbp[((pbp["pass"] == 1) | (pbp["rush"] == 1)) & pbp["epa"].notna()].copy()
print(pbp.shape, plays.shape)   

defense = plays.groupby(["game_id", "defteam"]).agg(
    def_epa=("epa", "mean"),    
).reset_index()

defense.shape 
defense.head()

(48771, 11) (36155, 11)


,game_id,defteam,def_epa
0,2025_01_ARI_NO,ARI,-0.057140
1,2025_01_ARI_NO,NO,0.061773
2,2025_01_BAL_BUF,BAL,0.256330
3,2025_01_BAL_BUF,BUF,0.394435
4,2025_01_CAR_JAX,CAR,0.103053


## 3. One row per team per game

In [43]:
offense = offense.rename(columns={"posteam": "team"})
defense = defense.rename(columns={"defteam": "team"})
combined = pd.merge(offense, defense, on=["game_id", "team"])
print(offense.shape, combined.shape)
combined.head()

(570, 6) (570, 7)


,game_id,season,week,team,off_epa,plays,def_epa
0,2025_01_ARI_NO,2025,1,ARI,0.061773,66,-0.057140
1,2025_01_ARI_NO,2025,1,NO,-0.057140,73,0.061773
2,2025_01_BAL_BUF,2025,1,BAL,0.394435,51,0.256330
3,2025_01_BAL_BUF,2025,1,BUF,0.256330,81,0.394435
4,2025_01_CAR_JAX,2025,1,CAR,-0.176198,63,0.103053


## 4. Season averages: a quick sanity check (lower defensive EPA = better defense)

In [44]:
season = combined.groupby("team")[["off_epa", "def_epa"]].mean()
season.sort_values("def_epa", ascending=True)

,off_epa,def_epa
team,,
HOU,-0.028835,-0.144154
SEA,0.063677,-0.138031
MIN,-0.114212,-0.093609
CLE,-0.185418,-0.081810
NE,0.102345,-0.081354
LAC,-0.031088,-0.077371
DEN,0.014545,-0.073634
JAX,0.041057,-0.073458
NO,-0.077004,-0.050166


## 5. Each team's games in order

In [45]:
combined = combined.sort_values(["team", "season", "week"]).reset_index(drop=True)
combined[combined["team"] == "SF"].head(10)

,game_id,season,week,team,off_epa,plays,def_epa
500,2025_01_SF_SEA,2025,1,SF,0.125300,74,-0.178174
501,2025_02_SF_NO,2025,2,SF,0.090516,67,-0.051837
502,2025_03_ARI_SF,2025,3,SF,-0.091097,63,0.045186
503,2025_04_JAX_SF,2025,4,SF,-0.054000,66,0.071395
504,2025_05_SF_LA,2025,5,SF,0.126715,85,0.098091
505,2025_06_SF_TB,2025,6,SF,-0.089480,69,0.266441
506,2025_07_ATL_SF,2025,7,SF,0.034464,64,-0.112618
507,2025_08_SF_HOU,2025,8,SF,-0.023637,48,0.207817
508,2025_09_SF_NYG,2025,9,SF,0.309834,64,0.172433
509,2025_10_LA_SF,2025,10,SF,0.112341,63,0.414148


## 6. Pre-game averages, with no leakage

`shift(1)` drops the current game before averaging, so a game's feature only uses earlier games:
week 1 has no prior (NaN), week 2 uses only week 1, and so on. This walkthrough averages the last
8 games; the pipeline uses 9 (`WINDOW` in `epa.py`).

In [46]:
combined["avg_off_epa_prior"] = (
    combined.groupby("team")["off_epa"]
    .transform(lambda s: s.shift(1).rolling(8, min_periods=1).mean())
)
combined[combined["team"] == "SF"][["week", "off_epa", "avg_off_epa_prior"]].head(5)

,week,off_epa,avg_off_epa_prior
500,1,0.125300,NaN
501,2,0.090516,0.125300
502,3,-0.091097,0.107908
503,4,-0.054000,0.041573
504,5,0.126715,0.017680


In [47]:
combined["avg_def_epa_prior"] = (
    combined.groupby("team")["def_epa"].transform(lambda s: s.shift(1).rolling(8, min_periods=1).mean())
)
combined[combined["team"] == "SF"][["week", "def_epa", "avg_def_epa_prior"]].head(5)

,week,def_epa,avg_def_epa_prior
500,1,-0.178174,NaN
501,2,-0.051837,-0.178174
502,3,0.045186,-0.115005
503,4,0.071395,-0.061608
504,5,0.098091,-0.028358
